# Joint representation of GEX and ADT data

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import warnings 
import numpy as np
import pandas as pd
import seaborn as sns
import muon as mu
import anndata as ad


warnings.filterwarnings('ignore')
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=200, facecolor='white')

#### Load in the protein and the rna data and check the names and edit them to include dataset name

In [ ]:
adata_protein = sc.read_h5ad("/home/prisb/projects/henry_hashtag_experiment/exmds/combined_protein.h5ad")

In [ ]:
adata_protein.obs.index = adata_protein.obs["dataset"].astype('str') + "_" + adata_protein.obs.index


In [ ]:
adata_protein.obs['dataset'].value_counts()

In [ ]:
adata_protein.obs_names.duplicated().sum()

In [ ]:
adata_rna = sc.read_h5ad("13052025_updated_annotations_colors_scores.h5ad")
adata_rna

In [ ]:
adata_rna.obs['timepoint_coarse'].value_counts()

In [ ]:
adata_rna.obs.index = adata_rna.obs["dataset_name"].astype('str') + "_" + adata_rna.obs.index

In [ ]:
adata_rna.obs_names.duplicated().sum()

In [ ]:
adata_rna.obs['patient_alias'].value_counts()

In [ ]:
#import anndata as ad

#X_cnv = adata_rna.obsm['X_cnv']
#obs = adata_rna.obs.copy()

#n_segments = X_cnv.shape[1]
#var = pd.DataFrame(index=[f"Segment_{i+1}" for i in range(n_segments)])

#adata_cnv = ad.AnnData(X_cnv, obs=obs, var=var)
#adata_cnv

#### Mutation data

In [ ]:
df_mutations=pd.read_csv("data/single_cell_mutation_for_priscilla.csv", index_col=0)
df_mutations

In [ ]:
df2 = pd.DataFrame(adata_rna.obs['BARCODE'])
df2

In [ ]:
df2['dataset_barcode'] = df2.index
df2

In [ ]:
df_matched = df_mutations.merge(df2[['BARCODE','dataset_barcode']], how='left', on='BARCODE').fillna('NA')


In [ ]:
df_matched.set_index('dataset_barcode', inplace=True)
df_matched

In [ ]:
c_dict = {'unknown':0, 'wt':1, 'mt':2}

mutations_df = df_matched.filter(like='call', axis=1).replace(c_dict).astype(float)
adata_mutations = ad.AnnData(X=np.array(mutations_df), obs=pd.DataFrame(index=mutations_df.index))
adata_mutations.var.index = [mutations_df.columns[i] for i in range(len(mutations_df.columns))]
adata_mutations

#### Merge the RNA and Protein adata objects

In [ ]:
mdata = mu.MuData({"rna":adata_rna, "protein":adata_protein})
mdata

In [ ]:
#mdata.mod['mutations'] = adata_mutations
#mdata

In [ ]:
mu.pp.intersect_obs(mdata)

In [ ]:
mdata.obs

In [ ]:
mdata['protein'].var["highly_variable"] = True   

In [ ]:
#del adata_cnv,adata_rna, adata_protein
#del obs,var,X_cnv
#import gc
#gc.collect()

#mdata.write("temp_15052025.h5mu")

In [ ]:
#Ran on katana for more memory space
#mu.tl.mofa(mdata, outfile="/models/mofa_model_15052025.hdf5", n_factors=15)

In [ ]:
mdata

In [ ]:
import mofax as mofa
model=mofa.mofa_model('mds_citeseq_2.hdf5')
model

In [ ]:
mdata

In [ ]:
sc.pp.neighbors(mdata, use_rep="X_mofa")

In [ ]:
sc.tl.umap(mdata)

In [ ]:
mdata.varm['LFs']

In [ ]:
mu.pl.mofa(mdata,color=['protein:ADT.CD105','protein:ADT.CD36'])

In [ ]:
mu.pl.umap(mdata, color=['rna:new_celltype','rna:outcome_C12D29','rna:outcome_C6D28'], ncols=2)

In [ ]:
mdata.write_h5mu("10_GEX_ADT_representation_DE.h5mu")

#### Interpret the integrated data

In [ ]:
import mofax as mofa

In [ ]:
sns.set_style("whitegrid")
mofa.plot_r2(model, x='View')


In [ ]:
mofa.plot_weights(model, views=['protein'], factors=0 , ncols=1, zero_line=True, label_size=10)

In [ ]:
mofa.plot_weights(model, views=['protein'], factors=1 , ncols=1, zero_line=True, label_size=10)

In [ ]:
mu.pl.umap(mdata, color=["ADT.CD36","ADT.CD105","ADT.CD7","ADT.HLA.ABC","ADT.CD33","ADT.CD11a","ADT.CD44","ADT.CD371","ADT.CD48"], ncols=3)

In [ ]:
mu.pl.mofa(mdata, color=["rna:outcome_C6D28","rna:outcome_C12D29","ADT.CD36","ADT.CD105","ADT.CD7","ADT.HLA.ABC","ADT.CD33","ADT.CD11a","ADT.CD44","ADT.CD371","ADT.CD48"], ncols=3)